# 🔍 Actividad 04: Calidad de los Datos
---
**Entrada:** CSVs de `data/02_interim/`  
**Salida:** `data/04_reports/reporte_calidad_datos.txt`

> Análisis visual de nulos, duplicados y outliers para las 3 fuentes del pipeline.


In [ ]:

import os, sys, json, glob, re, warnings, unicodedata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

# Navegar a la raíz del proyecto
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
print(f"Directorio: {os.getcwd()}")

with open('data/02_interim/pipeline_config.json', 'r', encoding='utf-8') as f:
    CFG = json.load(f)
DIRS = CFG['DIRS']
INTERIM = DIRS['interim']
REPORTS = DIRS['reports']
PROCESSED = DIRS['processed']
print("Configuración cargada OK")


## 4.1 Función de Diagnóstico de Calidad

In [ ]:

def quality_report(df, name, key_cols=None):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    
    # Nulos
    nulls = df.isnull().sum()
    nulls_pct = (nulls/len(df)*100).round(2)
    null_df = pd.DataFrame({'Nulos': nulls, '% Nulos': nulls_pct})
    null_df = null_df[null_df['Nulos'] > 0].sort_values('Nulos', ascending=False)
    
    if len(null_df) > 0:
        print(f"\n  Columnas con valores nulos ({len(null_df)}):")
        print(null_df.to_string())
    else:
        print("  ✅ Sin valores nulos.")
    
    # Duplicados
    if key_cols:
        dupes = df.duplicated(subset=key_cols).sum()
        print(f"\n  Duplicados (llave {key_cols}): {dupes}")
    else:
        dupes = df.duplicated().sum()
        print(f"\n  Duplicados totales: {dupes}")
    
    return null_df, dupes

df_m = pd.read_csv(f"{INTERIM}/midagri_limon_raw.csv")
df_ev = pd.read_csv(f"{INTERIM}/indeci_eventos_dbf.csv", low_memory=False)
df_n  = pd.read_csv(f"{INTERIM}/agraria_noticias_raw.csv")

null_m,  dup_m  = quality_report(df_m,  "MIDAGRI — midagri_limon_raw.csv",    ['anho','mes','COD_UBIGEO','dsc_Cultivo'])
null_ev, dup_ev = quality_report(df_ev, "INDECI — indeci_eventos_dbf.csv",     ['ide_sinpad'])
null_n,  dup_n  = quality_report(df_n,  "AGRARIA.PE — agraria_noticias_raw.csv", ['url'])


## 4.2 Visualización de Nulos por Fuente

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fuentes = [
    (df_m,  "MIDAGRI",    'mediumseagreen'),
    (df_ev, "INDECI",     'steelblue'),
    (df_n,  "AGRARIA.PE", 'coral'),
]

for ax, (df, nombre, color) in zip(axes, fuentes):
    nulls = (df.isnull().sum()/len(df)*100).sort_values(ascending=False)
    nulls = nulls[nulls > 0].head(15)
    if len(nulls) > 0:
        nulls.plot(kind='bar', ax=ax, color=color, edgecolor='black', linewidth=0.5)
        ax.set_title(f'{nombre}\n% Nulos por Columna', fontsize=11, fontweight='bold')
        ax.set_ylabel('% Nulos')
        ax.tick_params(axis='x', rotation=60)
        for p in ax.patches:
            if p.get_height() > 0:
                ax.text(p.get_x()+p.get_width()/2, p.get_height()+0.5,
                        f'{p.get_height():.1f}%', ha='center', fontsize=7)
    else:
        ax.text(0.5, 0.5, '✅ Sin nulos', ha='center', va='center',
                transform=ax.transAxes, fontsize=14, color='green', fontweight='bold')
        ax.set_title(f'{nombre}\nCalidad de Nulos', fontsize=11, fontweight='bold')

plt.suptitle('Análisis de Valores Nulos por Fuente de Datos', fontsize=14, fontweight='bold')
plt.tight_layout()
g_path = f"{REPORTS}/g4_calidad_nulos.png"
plt.savefig(g_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] {g_path}")


## 4.3 Análisis de Outliers — Producción MIDAGRI

In [ ]:

prod = pd.to_numeric(df_m['PRODUCCION(t)'], errors='coerce').dropna()
q1, q3 = prod.quantile(0.25), prod.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = prod[(prod < lower) | (prod > upper)]

print(f"Estadísticas de Producción (t):")
print(f"  Min:    {prod.min():>10.2f}")
print(f"  Q1:     {q1:>10.2f}")
print(f"  Median: {prod.median():>10.2f}")
print(f"  Q3:     {q3:>10.2f}")
print(f"  Max:    {prod.max():>10.2f}")
print(f"  IQR:    {iqr:>10.2f}")
print(f"  Outliers (1.5×IQR): {len(outliers):,} ({len(outliers)/len(prod)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(prod.clip(upper=prod.quantile(0.99)), patch_artist=True,
                boxprops=dict(facecolor='lightgreen'))
axes[0].set_title('Boxplot Producción (t)\n(truncado en p99)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Toneladas')
axes[0].axhline(upper, color='red', linestyle='--', label=f'Límite IQR: {upper:.0f}')
axes[0].legend()

prod.clip(upper=prod.quantile(0.99)).hist(bins=50, ax=axes[1], color='mediumseagreen', edgecolor='black')
axes[1].set_title('Distribución de Producción (t)\n(truncado en p99)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Toneladas'); axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
g_path2 = f"{REPORTS}/g5_outliers_produccion.png"
plt.savefig(g_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] {g_path2}")


## 4.4 Resumen de Calidad — Tabla de Decisiones

In [ ]:

# Tabla resumen de decisiones por fuente
resumen = {
    'Fuente': ['MIDAGRI', 'INDECI Eventos', 'AGRARIA.PE'],
    'Filas': [len(df_m), len(df_ev), len(df_n)],
    'Columnas': [df_m.shape[1], df_ev.shape[1], df_n.shape[1]],
    'Cols con Nulos': [len(null_m), len(null_ev), len(null_n)],
    'Duplicados': [dup_m, dup_ev, dup_n],
    'Acción': [
        'Renombrar cols + normalizar geo',
        'Filtrar fenómenos hidrometeorológicos + normalizar geo',
        'Limpiar HTML/URLs + normalizar texto'
    ]
}
df_resumen = pd.DataFrame(resumen)
print(df_resumen.to_string(index=False))

# Guardar reporte TXT
report_path = f"{REPORTS}/reporte_calidad_datos.txt"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("REPORTE DE CALIDAD DE DATOS — FASE 1\n")
    f.write("="*60+"\n")
    f.write(df_resumen.to_string(index=False))
    f.write("\n\nDecisiones de limpieza documentadas.\n")
print(f"\n[OK] {report_path}")
print("[ACTIVIDAD 04] COMPLETADA.")


## TODO: INTEGRACIÓN DATA NASA (COMPAÑERO)

En esta actividad, cuando integres datos climáticos, **valida los rangos físicos**:

| Variable | Rango Válido | Acción si fuera de rango |
|:---------|:-------------|:------------------------|
| T2M (°C) | -10 a 45 | Marcar como NaN |
| PRECTOTCORR (mm/día) | 0 a 500 | Marcar como NaN |
| RH2M (%) | 0 a 100 | Marcar como NaN |
| WS2M (m/s) | 0 a 50 | Marcar como NaN |

```python
df_nasa = pd.read_csv(f"{INTERIM}/nasa_clima_raw.csv")
print("NASA - nulos:", df_nasa.isnull().sum())
print("Temps fuera de rango:", ((df_nasa['T2M'] < -10) | (df_nasa['T2M'] > 45)).sum())
print("Precip negativas:",     (df_nasa['PRECTOTCORR'] < 0).sum())
```
